# Module 2 · Lesson 05: Prompt Evaluation

How do you know if your prompt is **good**? In production, we use **LLM-as-Judge**
to automatically evaluate prompt quality.

## What you will learn
1. Why prompt evaluation matters
2. **LLM-as-Judge** — using one model to evaluate another
3. Building a **scoring rubric**
4. **A/B testing** prompts
5. Batch evaluation for consistency

In [1]:
# ── Setup ──────────────────────────────────────────────
import os, json
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown
 
load_dotenv(Path.cwd().parent / ".env")
 
from openai import OpenAI
client = OpenAI()
 
def ask(prompt, system=None, temperature=0.7, max_tokens=400):
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(
        model="gpt-4o-mini", messages=msgs,
        temperature=temperature, max_tokens=max_tokens
    )
    return r.choices[0].message.content
 
print("✅ Ready")

✅ Ready


---
## 1. LLM-as-Judge Pattern

Use a **stronger model** (or the same model with a judge prompt) to score outputs.

```
Prompt A → Model → Response A ─┐
                                ├─→ Judge (LLM) → Score
Rubric ─────────────────────────┘
```

In [2]:
def evaluate_response(question: str, response: str, criteria: str) -> dict:
    """Use LLM to evaluate this AI response on a scale of 1-10"""
 
    judge_prompt = f"""Evaluate this AI response on a scale of 1-10.
 
Question: {question}
Response: {response}
 
Criteria: {criteria}
 
Return JSON: {{"score":<1-10>, "reasoning":"<brief explanation>"}}"""
 
    result = ask(judge_prompt, temperature=0)
 
    try:
        # Clean up markdown code blocks if present
        text = result.strip()
        if text.startswith("```"):
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        return json.loads(text.strip())
    except:
        return {"score": 0, "reasoning": result}
 
# Test it
question = "What is recursion in programming?"
response = ask(question)

In [3]:
eval_response = evaluate_response(
    question,
    response,
    "Accuracy, clarity, use of examples, appropriate length."
)

display(Markdown(f"### Response:\n{response}\n\n ### Evaluation:\n **Score:** {eval_response.get('score', 'N/A')}\n **Reasoning:**{eval_response.get('reasoning', 'N/A')}"))

### Response:
Recursion in programming is a technique where a function calls itself directly or indirectly in order to solve a problem. The key idea behind recursion is to break down a complex problem into simpler subproblems, which can be solved using the same approach.

### Key Components of Recursion:

1. **Base Case**: This is the condition under which the recursive function stops calling itself. It prevents infinite recursion and is essential for the function to eventually terminate.

2. **Recursive Case**: This is the part of the function that includes the recursive call. The function processes the input and calls itself with a modified argument, moving towards the base case.

### Example:

A classic example of recursion is calculating the factorial of a number \( n \). The factorial of \( n \) (denoted as \( n! \)) is defined as:

- \( n! = n \times (n-1)! \) for \( n > 0 \)
- \( 0! = 1 \) (base case)

Here’s a simple implementation in Python:

```python
def factorial(n):
    if n == 0:  # Base case
        return 1
    else:  # Recursive case
        return n * factorial(n - 1)

# Example usage
print(factorial(5))  # Output: 120
```

### Advantages of Recursion:

- **Simplicity**: Recursive solutions can be more straightforward and easier to understand than their iterative counterparts, especially for problems that naturally fit a recursive structure (like tree traversals or certain mathematical problems).
  
- **Code Reduction**: Recursive solutions can reduce the amount of code needed by eliminating the need for explicit stack management (like loops).

### Disadvantages of Recursion:

- **Performance**: Recursive calls can lead to overhead due to multiple function calls and may cause stack overflow if the recursion depth is too high.

- **Memory Usage**: Each recursive call consumes stack space, which

 ### Evaluation:
 **Score:** 9
 **Reasoning:**The response accurately defines recursion, clearly explains its key components, and provides a relevant example with code. It also discusses the advantages and disadvantages of recursion, which adds depth to the explanation. The only minor improvement could be a more concise conclusion or summary to enhance clarity.

---
## 2. A/B Testing Prompts

Compare two different prompts on the **same question**:

In [4]:
question = "Explain what Docker is to a junior developer"

# Simple prompt
response_a = ask(question)

# Prompt: PCTF
response_b = ask(
    question,
    system="You are a senior DevOps engineer. Use real-world analogy. Keep it under 100 words."
)

criteria = "Clarity for beginner, use of analogies, conciseness, actionable information."

eval_a = evaluate_response(question, response_a, criteria)
eval_b = evaluate_response(question, response_b, criteria)

# Display results
md = f"""### A/B Test Results
 
| | Prompt A (Simple) | Prompt B (PCTF) |
|---|---|---|
| **Score** | {eval_a.get('score', 'N/A')}/10 | {eval_b.get('score', 'N/A')}/10 |
| **Words** | {len(response_a.split())} | {len(response_b.split())} |
 
#### Prompt A Response
{response_a[:300]}{'...' if len(response_a)>300 else ''}
 
#### Prompt B Response
{response_b[:300]}{'...' if len(response_b)>300 else ''}
"""
display(Markdown(md))
 
winner = "A" if eval_a.get('score',0) > eval_b.get('score',0) else "B"
print(f"\n Winner: Prompt {winner}")

### A/B Test Results

| | Prompt A (Simple) | Prompt B (PCTF) |
|---|---|---|
| **Score** | 9/10 | 9/10 |
| **Words** | 318 | 88 |

#### Prompt A Response
Sure! Docker is a platform that helps developers build, package, and run applications in a more efficient and consistent way. Here’s a simple breakdown for you:

### What is Docker?

1. **Containerization**: At its core, Docker uses a technology called containerization. Think of a container as a lig...

#### Prompt B Response
Think of Docker like a shipping container for software. Just as containers allow goods to be transported easily across different ships, trains, and trucks without worrying about the contents, Docker packages your application and all its dependencies into a "container." This ensures it runs consisten...



 Winner: Prompt B


In [5]:
question = "Explain what Docker is to a junior developer."

# Simple prompt
response_a = ask(question)

# Prompt: PCTF
response_b = ask(
    question,
    system=("You are a senior DevOps engineer mentoring a junio engineer on their first day."
    "Use a single real-world analogy (shipping containers) to ground the explanation."
    "Keep your answer under 80 words."
    "End with one concrete terminal command the junior can try right now."
    )
)

criteria = (
    "Clarity for beginners (0-10)"
    "Quality of analogies (0-10)"
    "Concisceness - shorter is always better (0-10)"
    "Actionable next-step the reader can try immediately (0-10)"
    "Score each dimension then average for the final score"
)

eval_a = evaluate_response(question, response_a, criteria)
eval_b = evaluate_response(question, response_b, criteria)
 
words_a = len(response_a.split())
words_b = len(response_b.split())
 
winner = "A" if eval_a.get("score", 0) > eval_b.get("score", 0) else "B"
 
md = f"""
# A/B Test Results
 
## Question
**{question}**
 
| Metric | Prompt A (Simple) | Prompt B (PCTF) |
|---|---:|---:|
| **Overall Score** | {eval_a.get('score', 'N/A')}/10 | {eval_b.get('score', 'N/A')}/10 |
| **Clarity** | {eval_a.get('breakdown', {}).get('clarity', 'N/A')}/10 | {eval_b.get('breakdown', {}).get('clarity', 'N/A')}/10 |
| **Analogy** | {eval_a.get('breakdown', {}).get('analogy', 'N/A')}/10 | {eval_b.get('breakdown', {}).get('analogy', 'N/A')}/10 |
| **Conciseness** | {eval_a.get('breakdown', {}).get('conciseness', 'N/A')}/10 | {eval_b.get('breakdown', {}).get('conciseness', 'N/A')}/10 |
| **Actionable** | {eval_a.get('breakdown', {}).get('actionable', 'N/A')}/10 | {eval_b.get('breakdown', {}).get('actionable', 'N/A')}/10 |
| **Word Count** | {words_a} | {words_b} |
 
## Prompt A Response
{response_a[:500]}{'...' if len(response_a) > 500 else ''}
 
**Judge summary:** {eval_a.get('one_liner', 'N/A')}
 
---
 
## Prompt B Response
{response_b[:500]}{'...' if len(response_b) > 500 else ''}
 
**Judge summary:** {eval_b.get('one_liner', 'N/A')}
 
---
 
# 🏆 Winner: Prompt {winner}
"""
 
display(Markdown(md))


# A/B Test Results

## Question
**Explain what Docker is to a junior developer.**

| Metric | Prompt A (Simple) | Prompt B (PCTF) |
|---|---:|---:|
| **Overall Score** | 8/10 | 8/10 |
| **Clarity** | N/A/10 | N/A/10 |
| **Analogy** | N/A/10 | N/A/10 |
| **Conciseness** | N/A/10 | N/A/10 |
| **Actionable** | N/A/10 | N/A/10 |
| **Word Count** | 321 | 61 |

## Prompt A Response
Sure! Docker is a platform that helps developers build, package, and run applications in a consistent environment, regardless of where they are deployed. Here’s a breakdown of the key concepts:

1. **Containers**: At the heart of Docker is the concept of containers. A container is like a lightweight, portable, and self-sufficient unit that includes everything needed to run a piece of software — the code, runtime, libraries, and system tools. This means that you can run your application in the sa...

**Judge summary:** N/A

---

## Prompt B Response
Think of Docker like shipping containers for software. Just as shipping containers standardize how goods are transported, Docker packages applications and their dependencies into containers. This ensures they run consistently anywhere, just like containers can be transported across ships, trains, or trucks without issues. To get started, try running this command to see if Docker is installed: 

```bash
docker --version
```

**Judge summary:** N/A

---

# 🏆 Winner: Prompt B


---
## 3. Batch Evaluation

Test a prompt across **multiple questions** to measure consistency:

---
## Key Takeaways 📝

| Concept | Detail |
|---------|--------|
| **LLM-as-Judge** | Use an LLM to score other LLM outputs |
| **A/B testing** | Compare prompt variants on same questions |
| **Batch evaluation** | Test across multiple inputs for consistency |
| **Scoring rubric** | Define clear criteria (accuracy, clarity, etc.) |
| **JSON output** | Have the judge return structured scores |

---
**Next:** `06_output_parsing.ipynb` — Parse and structure LLM outputs reliably

In [6]:
# Prompts for evaluation
prompt1 = "Explain APIs."
prompt2 = "Explain what an API is and give an example."
prompt3 = "You are a software architect. Explain what an API is to non-technical business stakeholders. Use a real-world analogy and provide one practical example from web development. Structure your answer in bullet points."

# Responses for evaluation
response1 = ask(prompt1)
response2 = ask(prompt2)
response3 = ask(prompt3)

In [7]:
# Criteria
criteria = "Clarity, Context, Persona, Quality of response, Format"

In [8]:
def create_prompt_evaluation(prompt, criteria, response):
    """Create a prompt that asks for evaluation of a provided query and its response on specified criteria."""

    prompt_for_eval =f"""Evaluate this prompt based on the criteria provided.
 
    Prompt: {prompt}
 
    Criteria: {criteria}

    Response: {response}

    Format: For each criteria included, rate it from 1 to 5 and offer brief explanation.
    DO NOT evaluate the response. Evaluate ONLY the prompt. Use the response only as one of the criteria to evaluate the prompt.

    Example:
    Example Prompt: Explain Neural Networks.
    Example criteria: Persona, Format.
    Persona rating: 1, no persona details provided.
    Format rating: 1, no format details provided.
    """
    return prompt_for_eval

In [9]:
eval_response1 = ask(create_prompt_evaluation(prompt1, criteria, response1))
eval_response2 = ask(create_prompt_evaluation(prompt2, criteria, response2))
eval_response3 = ask(create_prompt_evaluation(prompt3, criteria, response3))

display(Markdown(eval_response1))
display(Markdown(eval_response2))
display(Markdown(eval_response3))

**Prompt Evaluation: Explain APIs**

**Clarity: 4**  
The prompt is clear in its intent, asking for an explanation of APIs. However, the term "APIs" could be unfamiliar to some audiences, which might require additional context for those who are not tech-savvy.

**Context: 3**  
The prompt lacks context regarding the audience or the level of detail expected. It does not specify whether the explanation should be technical or simplified for a general audience, which may lead to varied interpretations of the response.

**Persona: 2**  
The prompt does not establish a specific persona or target audience. Knowing whether the explanation is for developers, business professionals, or laypersons would help tailor the response more effectively.

**Quality of Response: 4**  
While the response is high-quality and informative, it is important to note that the quality of the response can only be fully assessed if the prompt provides clarity on the expected depth and complexity of the explanation.

**Format: 3**  
The prompt does not specify a format for the response, which could lead to inconsistent answers. A suggestion for structure (e.g., bullet points, examples, or sections) would enhance clarity and organization in the response.

**Overall Summary:**  
The prompt is effective but could benefit from more specificity regarding the audience and expectations for the response. More structured guidance on format would also improve the quality of responses.

**Prompt Evaluation: "Explain what an API is and give an example."**

1. **Clarity: 5**  
   The prompt is clear and straightforward, asking for an explanation of what an API is along with an example. There is no ambiguity in what is being requested.

2. **Context: 4**  
   The prompt provides sufficient context by specifying that the explanation should include an example. However, it could benefit from a bit more context about the audience or the purpose of the explanation (e.g., for beginners, developers, etc.).

3. **Persona: 3**  
   The prompt does not specify a particular persona or audience, which could help tailor the explanation. While it is broad enough to be applicable to many audiences, it lacks explicit guidance on who the explanation is intended for.

4. **Quality of Response: 4**  
   The prompt effectively leads to a high-quality response by asking for both an explanation and an example. This structure encourages a more in-depth answer, as demonstrated in the provided response.

5. **Format: 4**  
   The format is simple yet effective, asking for two components: an explanation and an example. However, it could be improved by explicitly stating how detailed the response should be or if there are specific aspects of APIs to focus on.

Overall, the prompt is effective but could be enhanced by providing more context and clarity regarding the intended audience and depth of response.

### Prompt Evaluation

**Prompt:** You are a software architect. Explain what an API is to non-technical business stakeholders. Use a real-world analogy and provide one practical example from web development. Structure your answer in bullet points.

#### Criteria Evaluation

- **Clarity: 5**
  - The prompt is clear and specific about what is expected. It defines the audience (non-technical business stakeholders) and the structure (bullet points), making it straightforward to understand the requirement.

- **Context: 5**
  - The context is well-defined as it addresses the need for non-technical stakeholders to understand a complex concept (API). It sets the stage for an explanation that is relatable and practical, which is essential for the intended audience.

- **Persona: 5**
  - The persona is appropriate; it specifies that the explanation should be tailored for non-technical business stakeholders. This helps the responder to focus on simplifying technical jargon and using relatable analogies.

- **Quality of response: 4**
  - While the quality of the expected response is implied in the prompt, it does not explicitly outline what constitutes a high-quality response. However, it does set expectations for using a real-world analogy and a practical example, which contributes positively to the anticipated quality.

- **Format: 5**
  - The prompt clearly specifies the use of bullet points to structure the response, which is a good choice for clarity and readability. This format allows for easy consumption of information, especially for non-technical stakeholders.

### Overall Ratings
- Clarity: 5
- Context: 5
- Persona: 5
- Quality of response: 4
- Format: 5